# XGBoost

- Read in tabular data
- Define predictor and target variables
- Train XGBoost model

In [1]:
import xgboost as xgb

from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt

import os
import sys
import yaml

sys.path.append('/home/548/cd3022/repos/solar-nowcast/modules')
import data_transform
from xgb_preprocess import prepare_data

from sklearn.metrics import mean_squared_error

In [2]:
config_name = 'vanilla'
# Get configurations from yaml file
with open(f"/home/548/cd3022/repos/solar-nowcast/configs/mouse/{config_name}.yaml") as f:
    config = yaml.safe_load(f)
# Model
model_name = config["model"]["name"]
forecast_lead = config["model"]["forecast_lead"]
# Parameters
random_state = config["model"]["parameters"]["random_state"]
n_estimators = config["model"]["parameters"]["n_estimators"]
early_stopping_rounds = config["model"]["parameters"]["early_stopping_rounds"]
learning_rate = config["model"]["parameters"]["learning_rate"]
eval_metric = config["model"]["parameters"]["eval_metric"]

# Data
X_vars = config["data"]["predictors"]
target = config["data"]["target"]
y_var = f'{target}_t{forecast_lead}'
all_vars = X_vars + [y_var]

In [3]:
# data_path = Path('/scratch/er8/cd3022/xgb_datasets/')

# df = pd.concat(
#     pd.read_parquet(f, columns=all_vars)
#     for f in data_path.glob("all_training_month*")
# )

In [3]:
data_path = Path('/scratch/er8/cd3022/xgb_datasets/')

train_files = []
test_files = []

for f in data_path.glob("all_training_month*"):
    month = f.name.split("_")[-1][:2]  # adjust to your naming

    if month in ['02', '06', '10']:
        test_files.append(str(f))
    else:
        train_files.append(str(f))

df_train = dd.read_parquet(train_files, columns=all_vars)
df_test  = dd.read_parquet(test_files, columns=all_vars)

In [4]:
X_train = df_train[X_vars]
y_train = df_train[y_var]

X_test = df_test[X_vars]
y_test = df_test[y_var]

In [5]:
# X_train, X_test, y_train, y_test = prepare_data(
#     df=df,
#     X=X_vars,
#     y=y_var,
#     test_months=['02', '06', '10']
# )

In [ ]:
# DEFINE MODEL
model = xgb.XGBRegressor(
    random_state=random_state,
    n_estimators=n_estimators,
    early_stopping_rounds=early_stopping_rounds,
    learning_rate=learning_rate,
    eval_metric=eval_metric,
)

# TRAINING
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=True
)

[0]	validation_0-rmse:0.25366	validation_1-rmse:0.26826
[1]	validation_0-rmse:0.24934	validation_1-rmse:0.26402
[2]	validation_0-rmse:0.24519	validation_1-rmse:0.25997
[3]	validation_0-rmse:0.24122	validation_1-rmse:0.25611
[4]	validation_0-rmse:0.23742	validation_1-rmse:0.25246
[5]	validation_0-rmse:0.23378	validation_1-rmse:0.24896
[6]	validation_0-rmse:0.23031	validation_1-rmse:0.24559
[7]	validation_0-rmse:0.22698	validation_1-rmse:0.24241
[8]	validation_0-rmse:0.22378	validation_1-rmse:0.23939
[9]	validation_0-rmse:0.22073	validation_1-rmse:0.23652
[10]	validation_0-rmse:0.21782	validation_1-rmse:0.23379
[11]	validation_0-rmse:0.21503	validation_1-rmse:0.23122
[12]	validation_0-rmse:0.21236	validation_1-rmse:0.22872
[13]	validation_0-rmse:0.20983	validation_1-rmse:0.22636
[14]	validation_0-rmse:0.20738	validation_1-rmse:0.22415
[15]	validation_0-rmse:0.20506	validation_1-rmse:0.22203
[16]	validation_0-rmse:0.20283	validation_1-rmse:0.22002
[17]	validation_0-rmse:0.20072	validation

In [ ]:
results = model.evals_result()

train_loss = results['validation_0']['rmse']
val_loss = results['validation_1']['rmse']

plt.figure()
plt.plot(train_loss, label='Train RMSE')
plt.plot(val_loss, label='Validation RMSE')

plt.xlabel('Boosting Iterations')
plt.ylabel('RMSE')
plt.title('Training vs Validation Loss')
plt.legend()

plt.show()

In [ ]:
# Predict on test set
y_pred = model.predict(X_test)

In [10]:
# Quick evaluation of model performance using correlation and RMSE
correlation = np.corrcoef(y_test['cloud_optical_depth_t1'], y_pred)[0, 1]
rmse = np.sqrt(mean_squared_error(y_test['cloud_optical_depth_t1'], y_pred))

print(f"Correlation between predicted and actual values: {correlation:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")

Correlation between predicted and actual values: 0.470
Root Mean Squared Error (RMSE): 17.127


In [ ]:
model.save_model(f"/scratch/er8/cd3022/xgb_models/{model_name}.json")